In [1]:
import mne
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import regex as reg
from sklearn.pipeline import Pipeline
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report

In [2]:
import warnings

mne.set_log_level('ERROR')   # silence MNE

warnings.filterwarnings("ignore")  # silence warnings

In [3]:
latest_channel_list = [
    # Left sensorimotor area channels
    'E29', 'E30', 'E35', 'E36', 'E41', 'E42',
    # Right sensorimotor area channels
    'E103', 'E104', 'E109', 'E110', 'E115', 'E116',
    # Mid-parietal & bilateral parietal
    'E62', 'E67', 'E72', 'E77'
 ]

bad_channels = ['E48', 'E119', 'E49', 'E113', 'E94', 'E68', 'E23', 'E3', 'E126', 'E127']

channel_tuple = (latest_channel_list, bad_channels)

label_dict = {'OSBA': 0, 'OSBY': 1, 'OSDO': 2, 'OSMO': 3, 'OSSI':4}

directions = ['OSBA', 'OSBY', 'OSDO', 'OSDO','OSSI']  # Left, Right, Up, Down

# Gemini code:

In [4]:
class preprocessing_pipeline:
    def __init__(self, filename, *channel_tuple, l_freq=8.0, h_freq=30.0, notch_freq=50.0, fs=500.0, time_window=0.4):
        self.filename = filename
        self.l_freq = l_freq
        self.h_freq = h_freq
        self.notch_freq = notch_freq
        self.time_window = time_window
        self.fs = fs
        # Safety: check if channel_tuple has enough elements
        self.active_channels = channel_tuple[0] if len(channel_tuple) > 0 else []
        self.bad_channels = channel_tuple[1] if len(channel_tuple) > 1 else []

        self.raw = self.file_process()
        self.annotations = self.raw.annotations

    def file_process(self):
        raw = mne.io.read_raw_egi(self.filename, preload=True, verbose=False)
        if 'VREF' in raw.ch_names:
            raw.drop_channels(['VREF'])
        
        raw.pick('eeg')
        if self.bad_channels:
            raw.drop_channels([ch for ch in self.bad_channels if ch in raw.ch_names])
            
        raw.set_eeg_reference('average', projection=False)
        raw.filter(l_freq=self.l_freq, h_freq=self.h_freq, phase='zero', verbose=False)
        raw.notch_filter(freqs=self.notch_freq, verbose=False)

        # ICA Block
        rank = mne.compute_rank(raw, tol=1e-6, tol_kind='relative')
        n_components = min(25, rank['eeg'])
        ica = mne.preprocessing.ICA(n_components=n_components, random_state=42, method='fastica', max_iter='auto')
        ica.fit(raw, verbose=False)
        
        bad_components = []
        try:
            muscle_idx, _ = ica.find_bads_muscle(raw, threshold=0.5, verbose=False)
            bad_components.extend(muscle_idx)
        except: pass
            
        ica.exclude = list(set(bad_components))
        raw = ica.apply(raw, picks='eeg', verbose=False)
        return raw

    def extracting_data(self, label_dict):
        classes = ['BA', 'BY', 'DO', 'MO', 'SI']
        X_all, y_all = [], []

        for cls in classes:
            # Ensure onsets are handled as floats for cropping
            starts = [float(ann['onset']) for ann in self.annotations if ann['description'] == f'OS{cls}']
            ends = [float(ann['onset']) for ann in self.annotations if ann['description'] == f'OE{cls}']
            
            for start, end in zip(starts, ends):
                segment = self.raw.copy().crop(tmin=start, tmax=end)
                data = segment.get_data() 

                # FORCE integers for slicing
                window_samples = int(self.time_window * self.fs)
                total_samples = data.shape[1]
                num_windows = int(total_samples // window_samples)

                if num_windows <= 0:
                    continue

                for w in range(num_windows):
                    s_idx = int(w * window_samples)
                    e_idx = int(s_idx + window_samples)
                    X_all.append(data[:, s_idx:e_idx])
                    
                    # Force the label to be an integer scalar
                    label_val = int(label_dict[f'OS{cls}'])
                    y_all.append(label_val)

        if not X_all:
            return None, None

        return np.array(X_all), np.array(y_all)

In [ ]:
# # Making the dictionary for subject dirs
# total_subjects = 1
# subject_dirs = {}
# #base_dir = r'D:\\0001_AK\\KRISHNA\\001_EEG_work_recent\\extracted_files'
# base_dir = "/Users/kavinfidel/Desktop/GNN+CNS+Hopf/CNS_Lab/VM_EEG/Data/S_13"


# for i in range(1, total_subjects + 1):
#     files = []
#     for file_name in os.listdir(base_dir):
#         #m = reg.match(r'[a-zA-Z]ursor_S(\d+)_S\d+_B\d+', file_name)
#         if not file_name.startswith('.') and file_name.endswith('.mff'):
#             files.append(file_name)
#     subject_dirs[f'Subject_{i}'] = files


In [5]:
import os

# Point this to the parent "Data" directory
base_dir = "/Users/kavinfidel/Desktop/GNN+CNS+Hopf/CNS_Lab/VM_EEG/Data"

subject_dirs = {}

# 1. Get all items in the Data folder
# 2. Filter for directories that start with 'S'
sub_folders = [f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f)) and f.startswith('S')]

for folder in sub_folders:
    folder_path = os.path.join(base_dir, folder)
    files = []
    
    # List all .mff files within each subject's folder
    for file_name in os.listdir(folder_path):
        if not file_name.startswith('.') and file_name.endswith('.mff'):
            files.append(file_name)
    
    # Using the actual folder name (e.g., 'S1', 'S113') as the key
    subject_dirs[folder] = files

# Verification
print(f"Found {len(subject_dirs)} subjects.")
print("Subjects identified:", list(subject_dirs.keys()))

Found 13 subjects.
Subjects identified: ['S116', 'S118', 'S5', 'S2', 'S119', 'S117', 'S3', 'S4', 'S1', 'S6', 'S115', 'S113', 'S114']


In [6]:
total_data = {}
test_data = {}
#base_dir = r'D:\0001_AK\KRISHNA\001_EEG_work_recent\extracted_files'
base_dir = "/Users/kavinfidel/Desktop/GNN+CNS+Hopf/CNS_Lab/VM_EEG/Data"

In [7]:
for subject, files in subject_dirs.items():
    print(f"Processing {subject}")
    
    total_data[subject] = {}
    test_data[subject] = {}
    signals, labels = [], []
    signals_test, labels_test = [], []
    
    for k, file_name in enumerate(files, 1):
        file_path = os.path.join(base_dir, subject, file_name)
        
        # Checking for the .mff folder structure
        if not file_name.endswith('.mff') or not os.path.exists(os.path.join(file_path, "signal1.bin")):
            continue
        
        print(f"File is intact: {file_name}")
        
        try:
            processor = preprocessing_pipeline(file_path, *channel_tuple)
            # Pass the label_dict you defined elsewhere
            X, y = processor.extracting_data(label_dict) 
            
            if X is not None:
                if k == 2: # Your logic for testing block
                    signals_test.append(X)
                    labels_test.append(y)
                else:
                    signals.append(X)
                    labels.append(y)
        except Exception as e:
            print(f"Error processing {file_name}: {e}")
            continue
    
    # Safety check before concatenating
    if signals:
        total_data[subject]['data'] = np.concatenate(signals, axis=0)
        total_data[subject]['labels'] = np.concatenate(labels, axis=0)
    
    if signals_test:
        test_data[subject]['data'] = np.concatenate(signals_test, axis=0)
        test_data[subject]['labels'] = np.concatenate(labels_test, axis=0)

Processing S116
File is intact: VI_S6_S1_B3__20251116_110436.mff
Error processing VI_S6_S1_B3__20251116_110436.mff: apply() got an unexpected keyword argument 'picks'
File is intact: VI_S6_S1_B1__20251116_104819.mff
Error processing VI_S6_S1_B1__20251116_104819.mff: apply() got an unexpected keyword argument 'picks'
File is intact: VI_S6_S1_B2__20251116_105637.mff
Error processing VI_S6_S1_B2__20251116_105637.mff: apply() got an unexpected keyword argument 'picks'
Processing S118
File is intact: VI_S8_S1_B1__20251119_052228.mff
Error processing VI_S8_S1_B1__20251119_052228.mff: apply() got an unexpected keyword argument 'picks'
File is intact: VI_S8_S1_B2__20251119_053014.mff
Error processing VI_S8_S1_B2__20251119_053014.mff: apply() got an unexpected keyword argument 'picks'
File is intact: VI_S8_S1_B3__20251119_053757.mff
Error processing VI_S8_S1_B3__20251119_053757.mff: apply() got an unexpected keyword argument 'picks'
Processing S5
File is intact: S5_S1_B3_1654_161025_20251016_04

KeyboardInterrupt: 

In [ ]:
total_data = {}
test_data = {}
#base_dir = r'D:\0001_AK\KRISHNA\001_EEG_work_recent\extracted_files'
base_dir = "/Users/kavinfidel/Desktop/GNN+CNS+Hopf/CNS_Lab/VM_EEG/Data"

for subject, files in subject_dirs.items(): # subject is id, files are the all the files associated with a subject
    print(f"Processing {subject}")
    
    total_data[f"{subject}"] = {} #?
    test_data[f"{subject}"] = {}
    signals = [] #?
    labels = []#?
    signals_test = []
    labels_test = []
    k = 0
    for file_name in files:
        k +=1
        file_path = os.path.join(base_dir,subject, file_name) # grabbing file path, the mff file?
        
        if not file_name.endswith('.mff'):
            print(f"Skipping non-raw file: {file_name}")
            continue
        
        required_parts = ["signal1.bin", "info1.xml"]
        missing_parts = [p for p in required_parts if not os.path.exists(os.path.join(file_path, p))] # wha tis happenign here?
        if missing_parts:
            print(f"Skipping {file_name} due to parts being missing")
            continue
        
        print(f"File is intact: {file_name}\n Beginning extraction...")
        
        try:
            # if k==3 or subject == "S114" and k =2:
            if k ==2:
                print("3rd block is for testing")
                processor = preprocessing_pipeline(file_path, *channel_tuple)
                X,y = processor.extracting_data()
                signals_test.append(X)
                labels_test.append(y)
        
            else:
                print("file path",file_path)
                processor = preprocessing_pipeline(file_path, *channel_tuple)
                X,y = processor.extracting_data()
                signals.append(X)
                labels.append(y)
        except Exception as e:
            print(f"Error processing {file_name}: {e}")
            continue
    
    total_data[f"{subject}"]['data'] = np.concatenate(signals, axis=0)
    total_data[f"{subject}"]['labels'] = np.concatenate(labels, axis=0)   
    
    test_data[f"{subject}"]['data'] = np.concatenate(signals_test, axis=0)
    test_data[f"{subject}"]['labels'] = np.concatenate(labels_test, axis=0)  

    

In [ ]:
for subject in total_data.keys():
    data = total_data[subject]['data']
    labels = total_data[subject]['labels']
    
    print(f"--- Verification for {subject} ---")
    print(f"Data Shape: {data.shape}") 
    # Expected: (Total_Windows, Channels, Samples_per_Window)
    # Example: (1500, 128, 50) 
    
    print(f"Labels Shape: {labels.shape}")
    print(f"Unique Labels: {np.unique(labels)}")
    
    # Check if classes are balanced (should be roughly equal)
    unique, counts = np.unique(labels, return_counts=True)
    print("Samples per class:", dict(zip(unique, counts)))
    print("-" * 30)

In [ ]:
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

In [ ]:
def riemannian_model_build(X_train,y_train,X_test, clf_type):
    cov_est = Covariances(estimator='oas')
    ts = TangentSpace(metric='riemann')
    scaler = StandardScaler()
    
    # step 01
    cov_train = cov_est.fit_transform(X_train)
    cov_test = cov_est.transform(X_test)
    
    # Step 02
    X_train_ts = ts.fit_transform(cov_train)
    X_test_ts = ts.transform(cov_test)
    
    # Step 03
    X_train_scaled = scaler.fit_transform(X_train_ts)
    X_test_scaled = scaler.transform(X_test_ts)
    
    if clf_type == 'logreg':
        clf = LogisticRegression(
            penalty='l2',
            solver='lbfgs',
            class_weight='balanced',
            max_iter=1000
        )
    
    elif clf_type == 'svm':
        clf =   SVC(
            kernel='linear',
            class_weight='balanced',
            C=1.0
        )
    # elif clf_type == 'xgboost':
    #     # Ensure y_train is 0-indexed for XGBoost
    #     clf = XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3)
        
    # elif clf_type == 'mlp':
    #     clf = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, early_stopping=True)
    else:
        raise ValueError('No classifier model supported')
    
    clf.fit(X_train_scaled, y_train)
    y_pred = clf.predict(X_test_scaled)
    
    return y_pred        

In [ ]:
def model_performance(data_dict,clf_type):
    total_acc_results = {}
    
    for subject in list(data_dict.keys()):
        print(f"Processing {subject}...")
        X = data_dict[subject]['data']
        y = data_dict[subject]['labels']
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        acc_scores = []
        
        for train_index, test_index in skf.split(X, y):
            X_train, X_test = X[train_index], X[test_index]
            y_train, y_test = y[train_index], y[test_index]
            
            y_pred = riemannian_model_build(X_train, y_train, X_test, clf_type)
            acc = accuracy_score(y_test, y_pred)
            acc_scores.append(acc)
            print(f"Test Accuracy: {acc:.4f}")

        total_acc_results[f"{subject}"] = acc_scores
        print(f"{subject} - Average Accuracy: {np.mean(acc_scores):.4f} with Standard Deviation: {np.std(acc_scores):.4f}\n")
    
    return total_acc_results

## model performance with confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
import matplotlib.pyplot as plt

In [ ]:


def model_performance(data_dict,clf_type, class_names=['BA', 'BY', 'DO', 'MO', 'SI']):
    total_acc_results = {}
    
    for subject in list(data_dict.keys()):
        print(f"--- Processing {subject} ---")
        X = data_dict[subject]['data']
        y = data_dict[subject]['labels']
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        
        # Lists to store metrics for the whole subject
        acc_scores = []
        all_y_test = []
        all_y_pred = []
        
        for train_index, test_index in skf.split(X, y):
            X_train, X_test = X[train_index], X[test_index]
            y_train, y_test = y[train_index], y[test_index]
            
            # Predict
            y_pred = riemannian_model_build(X_train, y_train, X_test, clf_type)
            
            # Store for metrics
            acc = accuracy_score(y_test, y_pred)
            acc_scores.append(acc)
            
            # Append for Confusion Matrix
            all_y_test.extend(y_test)
            all_y_pred.extend(y_pred)

        # 1. Calculate Average Stats
        total_acc_results[f"{subject}"] = acc_scores
        print(f"Average Accuracy: {np.mean(acc_scores):.4f} +/- {np.std(acc_scores):.4f}")

        # 2. Plot Confusion Matrix
        cm = confusion_matrix(all_y_test, all_y_pred, normalize='true')
        
        # Plotting
        fig, ax = plt.subplots(figsize=(8, 6))
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
        disp.plot(cmap='Blues', ax=ax, values_format='.2g')
        plt.title(f"Confusion Matrix: {subject} ({clf_type})")
        plt.savefig(f"/Users/kavinfidel/Desktop/GNN+CNS+Hopf/CNS_Lab/VM_EEG/confusion_matrices/{clf_type}_{subject}_0.4.png")
        plt.show()
    
    return total_acc_results

In [ ]:
def model_performance(train_dict, test_dict, clf_type, class_names=['BA', 'BY', 'DO', 'MO', 'SI']):
    total_acc_results = {}
    
    # We iterate over subjects in the training dictionary
    for subject in train_dict.keys():
        # Safety check: make sure this subject actually has a test block
        if subject not in test_dict or 'data' not in test_dict[subject]:
            print(f"Skipping {subject}: No test data found.")
            continue

        print(f"--- Evaluating {subject} ---")
        
        # 1. Prepare Training Data (all blocks except the 3rd)
        X_train = train_dict[subject]['data']
        y_train = train_dict[subject]['labels']
        
        # 2. Prepare Test Data (the 3rd block we kept separate)
        X_test = test_dict[subject]['data']
        y_test = test_dict[subject]['labels']
        
        print(f"Train samples: {len(X_train)} | Test samples: {len(X_test)}")

        # 3. Predict 
        # We don't need the loop for K-Fold anymore because we have a fixed test set
        y_pred = riemannian_model_build(X_train, y_train, X_test, clf_type)
        
        # 4. Calculate Accuracy
        acc = accuracy_score(y_test, y_pred)
        total_acc_results[subject] = acc
        print(f"Final Test Accuracy: {acc:.4f}")

        # 5. Plot Confusion Matrix for this specific test block
        cm = confusion_matrix(y_test, y_pred, normalize='true')
        
        fig, ax = plt.subplots(figsize=(8, 6))
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
        disp.plot(cmap='Blues', ax=ax, values_format='.2g')
        plt.title(f"Hold-out Test: {subject} ({clf_type})")
        
        # Save and Show
        save_path = f"/Users/kavinfidel/Desktop/GNN+CNS+Hopf/CNS_Lab/VM_EEG/confusion_matrices/holdout_{clf_type}_{subject}.png"
        plt.savefig(save_path)
        plt.show()
    
    return total_acc_results

In [ ]:
xgb_acc_dict = model_performance(total_data, clf_type='xgboost')


In [ ]:

logreg_acc_dict = model_performance(total_data,test_data, clf_type='logreg')

In [ ]:
import json

# Save the dictionary to a file
with open('logreg_acc_results_0.4_chunk.json', 'w') as f:
    json.dump(logreg_acc_dict, f, indent=4)

print("Results saved to logreg_acc_results.json")

# To load it back later:
# with open('logreg_acc_results.json', 'r') as f:
#     loaded_results = json.load(f)

In [ ]:
svm_acc_dict = model_performance(total_data, clf_type='svm')

In [ ]:

with open('svm_acc_results_0.5_chunk.json', 'w') as f:
    json.dump(svm_acc_dict, f, indent=4)

In [ ]:
subject_labels = list(logreg_acc_dict.keys())

plt.figure(figsize=(25, 8))
bars = plt.bar(subject_labels, [np.mean(logreg_acc_dict[subj]) for subj in subject_labels], color='skyblue')
plt.xlabel('Test Subject (Fold)')
plt.ylabel('Test Accuracy')
plt.title('Test Accuracy for Each Subject (within subjects): Logistic Regression')
plt.bar_label(bars, padding=2)
plt.show()

In [ ]:
subject_labels = list(logreg_acc_dict.keys())

for fold_num in range(5):
     fold_accuracies = [logreg_acc_dict[subj][fold_num] for subj in subject_labels]
     plt.figure(figsize=(15, 5))
     bars = plt.bar(subject_labels, fold_accuracies, color='skyblue')
     plt.xlabel('Test Subject (Fold)')
     plt.xticks(rotation=45, ha='right')
     plt.ylabel('Test Accuracy')
     plt.title(f'Test Accuracy for Each Subject (within subjects): Logistic Regression - Fold {fold_num + 1}')
     plt.bar_label(bars, padding=2)
     plt.ylim(0, max(fold_accuracies) + 0.05)  # Set y-axis limits to [0, 1] for better visualization
     plt.show()

In [ ]:
subject_labels = list(svm_acc_dict.keys())

plt.figure(figsize=(25, 8))
bars = plt.bar(subject_labels, [np.mean(svm_acc_dict[subj]) for subj in subject_labels], color='skyblue')
plt.xlabel('Test Subject (Fold)')
plt.ylabel('Test Accuracy')
plt.title('Test Accuracy for Each Subject (within subjects): SVM')
plt.bar_label(bars, padding=2)
plt.show()

In [ ]:
subject_labels = list(svm_acc_dict.keys())

for fold_num in range(5):
     fold_accuracies = [svm_acc_dict[subj][fold_num] for subj in subject_labels]
     plt.figure(figsize=(15, 5))
     bars = plt.bar(subject_labels, fold_accuracies, color='skyblue')
     plt.xlabel('Test Subject (Fold)')
     plt.xticks(rotation=45, ha='right')
     plt.ylabel('Test Accuracy')
     plt.title(f'Test Accuracy for Each Subject (within subjects): SVM - Fold {fold_num + 1}')
     plt.bar_label(bars, padding=2)
     plt.ylim(0, max(fold_accuracies) + 0.05)  # Set y-axis limits to [0, 1] for better visualization
     plt.show()